# 05 - Métodos de ponto fixo
Vamos aprender sobre como usar os métodos de ponto fixo para encontrar as raízes de funções reais.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana5
```

Faça o checkout nessa nova branch:

```bash
git checkout semana5
```

Instale as bibliotecas NumPy e SciPy.

In [1]:
%pip install numpy scipy

Dependências já instaladas no ambiente de execução.


<hr />

## Atividade 1
Escreva os métodos de ponto fixo dentro do arquivo $utils/algoritmos.py$.

In [1]:
import math
import sys

def bissecao(f, a, b, TOL, iter=100):
    """Encontra uma raiz em [a, b] pelo método da bisseção."""
    if TOL <= 0:
        raise ValueError("A tolerância deve ser positiva.")
    max_iter = int(iter)
    if max_iter <= 0:
        raise ValueError("O número máximo de iterações deve ser positivo.")

    fa = f(a)
    fb = f(b)
    if fa == 0:
        return a, 0
    if fb == 0:
        return b, 0
    if fa * fb > 0:
        raise ValueError("Nenhuma raiz encontrada no intervalo.")

    c = (a + b) / 2.0
    for i in range(1, max_iter + 1):
        c = (a + b) / 2.0
        fc = f(c)
        if fc == 0 or abs(b - a) / 2.0 <= TOL:
            return c, i
        if fa * fc < 0:
            b, fb = c, fc
        else:
            a, fa = c, fc

    return c, max_iter

def pontofixo(a, g, TOL=1e-8, iter=1000):
    """Executa x_(n+1)=g(x_n) até estabilizar."""
    if TOL <= 0:
        raise ValueError("A tolerância deve ser positiva.")

    x = float(a)
    for i in range(1, int(iter) + 1):
        next_x = float(g(x))
        if not math.isfinite(next_x):
            raise ValueError("A iteração produziu um valor não finito.")
        if abs(next_x - x) <= TOL * max(1.0, abs(next_x)):
            return next_x, i
        x = next_x
    raise RuntimeError("O método do ponto fixo não convergiu.")

def newton_raphson(a, f, TOL=1e-8, df=None, iter=100):
    """Método de Newton-Raphson com derivada analítica ou diferença central."""
    def numerical_derivative(x):
        h = math.sqrt(sys.float_info.epsilon) * max(1.0, abs(x))
        return (f(x + h) - f(x - h)) / (2.0 * h)

    derivative = df or numerical_derivative

    def iteration(x):
        slope = derivative(x)
        if abs(slope) <= sys.float_info.epsilon:
            raise ZeroDivisionError("Derivada nula ou muito próxima de zero.")
        return x - f(x) / slope

    return pontofixo(a, iteration, TOL, iter)

def secante(a, b, f, TOL=1e-8, iter=100):
    """Método da secante com limite de iterações e teste de denominador."""
    x0, x1 = float(a), float(b)
    f0, f1 = f(x0), f(x1)

    for i in range(1, int(iter) + 1):
        denominator = f1 - f0
        if abs(denominator) <= sys.float_info.epsilon:
            raise ZeroDivisionError("Denominador nulo ou muito próximo de zero.")
        x2 = x1 - f1 * (x1 - x0) / denominator
        if not math.isfinite(x2):
            raise ValueError("A iteração produziu um valor não finito.")
        if abs(x2 - x1) <= TOL * max(1.0, abs(x2)):
            return x2, i
        x0, x1 = x1, x2
        f0, f1 = f1, f(x1)

    raise RuntimeError("O método da secante não convergiu.")



Resolver a equação $e^x = x + 2$ é equivalente a calcular os pontos fixos da função  

$$
g(x) = e^x - 2
$$

Use os métodos do ponto fixo  

$$
x^{(n+1)} = g(x^{(n)})
$$

com $x^{(0)} = -1.8$ para obter uma aproximação de uma das soluções da equação dada com 8 dígitos significativos.  

**Resposta:**  
$
x \approx -1.8414057
$

In [1]:
import math
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..")))
from utils.algoritmos import pontofixo, newton_raphson, secante

f1 = lambda x: math.exp(x) - x - 2
g1 = lambda x: math.exp(x) - 2

for name, result in [
    ("ponto fixo", pontofixo(-1.8, g1)),
    ("Newton-Raphson", newton_raphson(-1.8, f1, df=lambda x: math.exp(x)-1)),
    ("secante", secante(-1.8, -1.7, f1)),
]:
    print(f"{name}: x={result[0]:.8f}, iterações={result[1]}")


ponto fixo: x=-1.84140566, iterações=9
Newton-Raphson: x=-1.84140566, iterações=3
secante: x=-1.84140566, iterações=3


## Atividade 2
Encontre a raiz positiva da função  

$$
f(x) = \cos(x) - x^2
$$  

pelos métodos do ponto fixo, inicializando-o com $x^{(0)} = 1$.  

Realize a iteração até obter estabilidade no **quinto dígito significativo**.  

**Resposta:**

$$
x \approx 0.82413 
$$

Processo iterativo:  

$$
x^{(n+1)} = x^{(n)} + \frac{\cos(x) - x^2}{\sin(x) + 2x}
$$

In [1]:
f2 = lambda x: math.cos(x) - x*x
g2 = lambda x: x + (math.cos(x)-x*x)/(math.sin(x)+2*x)
df2 = lambda x: -math.sin(x) - 2*x

for name, result in [
    ("ponto fixo", pontofixo(1.0, g2, TOL=5e-6)),
    ("Newton-Raphson", newton_raphson(1.0, f2, TOL=5e-6, df=df2)),
    ("secante", secante(0.8, 1.0, f2, TOL=5e-6)),
]:
    print(f"{name}: x={result[0]:.5f}, iterações={result[1]}")


ponto fixo: x=0.82413, iterações=4
Newton-Raphson: x=0.82413, iterações=4
secante: x=0.82413, iterações=3


## Atividade 3
Aplique os métodos do ponto fixo para resolver a equação:

$$
e^{-x^2} = 2x
$$

**Resposta:**

$$
x \approx 0.4193648
$$

In [1]:
f3 = lambda x: math.exp(-x*x) - 2*x
g3 = lambda x: 0.5 * math.exp(-x*x)
df3 = lambda x: -2*x*math.exp(-x*x) - 2

for name, result in [
    ("ponto fixo", pontofixo(0.5, g3)),
    ("Newton-Raphson", newton_raphson(0.5, f3, df=df3)),
    ("secante", secante(0.4, 0.5, f3)),
]:
    print(f"{name}: x={result[0]:.8f}, iterações={result[1]}")


ponto fixo: x=0.41936482, iterações=17
Newton-Raphson: x=0.41936482, iterações=4
secante: x=0.41936482, iterações=3


## Atividade 4
Resolva os três últimos exercícios da semana anterior usando os métodos do ponto fixo.

In [1]:
# Atividade 4: exercícios 4, 5 e 6 da semana anterior.

def diode_equation(vd, voltage, resistance):
    reverse_current = 1e-12
    thermal_voltage = 1.38064852e-23 * 300.0 / 1.60217662e-19
    return resistance*reverse_current*math.expm1(vd/thermal_voltage) + vd - voltage

cases = [
    (30.0, 1e3, 0.5, 0.7), (3.0, 1e3, 0.4, 0.7),
    (3.0, 1e4, 0.4, 0.6), (0.3, 1e3, 0.2, 0.4),
    (-0.3, 1e3, -0.4, -0.2), (-30.0, 1e3, -31.0, -29.0),
    (-30.0, 1e4, -31.0, -29.0),
]
print("Diodo — Newton-Raphson")
for voltage, resistance, a, b in cases:
    thermal_voltage = 1.38064852e-23 * 300.0 / 1.60217662e-19
    derivative = lambda x: resistance * 1e-12 * math.exp(x / thermal_voltage) / thermal_voltage + 1.0
    root, iterations = newton_raphson((a + b) / 2, lambda x: diode_equation(x, voltage, resistance), df=derivative)
    print(f"V={voltage:g} V, R={resistance/1e3:g} kΩ: vd={root:.3f} V ({iterations} it.)")

def catenary(c):
    return c*math.cosh(500.0/(2*c)) - c - 50.0
def catenary_derivative(c):
    z = 250.0/c
    return math.cosh(z) - z*math.sinh(z) - 1.0

root_n, it_n = newton_raphson(600.0, catenary, df=catenary_derivative)
root_s, it_s = secante(550.0, 650.0, catenary)
print(f"Catenária — Newton: C={root_n:.4f} m; secante: C={root_s:.4f} m")

frequency, inductance, resistance = 1e3, 100e-3, 1e3
tan_phi = 2*math.pi*frequency*inductance/resistance
phi = math.atan(tan_phi)
def rectifier(beta):
    return math.sin(beta-phi) + math.sin(phi)*math.exp(-beta/tan_phi)
def rectifier_derivative(beta):
    return math.cos(beta-phi) - math.sin(phi)*math.exp(-beta/tan_phi)/tan_phi

a, b = math.radians(212), math.radians(213)
root_n, it_n = newton_raphson((a+b)/2, rectifier, df=rectifier_derivative)
root_s, it_s = secante(a, b, rectifier)
print(f"Retificador — Newton: β={math.degrees(root_n):.4f}°; secante: β={math.degrees(root_s):.4f}°")


Diodo — Newton-Raphson
V=30 V, R=1 kΩ: vd=0.623 V (6 it.)
V=3 V, R=1 kΩ: vd=0.559 V (5 it.)
V=3 V, R=10 kΩ: vd=0.500 V (3 it.)
V=0.3 V, R=1 kΩ: vd=0.300 V (2 it.)
V=-0.3 V, R=1 kΩ: vd=-0.300 V (1 it.)
V=-30 V, R=1 kΩ: vd=-30.000 V (1 it.)
V=-30 V, R=10 kΩ: vd=-30.000 V (1 it.)
Catenária — Newton: C=633.1622 m; secante: C=633.1622 m
Retificador — Newton: β=212.2258°; secante: β=212.2258°


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 5"
git push origin semana5
```